# 임베딩 매칭 vs 태그 매칭 — 품질 비교 (Kaggle)

이 노트북은 두 질문에 답한다.

1. **임베딩 매칭이 현재 태그 매칭보다 나은가?** — `OutfitMatcher`가 지금 쓰는
   색상/격식 태그 기반 점수와, 이미지 임베딩 코사인 유사도를 나란히 놓고 비교한다.
2. **패션 특화 모델이 값을 하는가?** — 범용 CLIP(OpenCLIP) vs 패션 특화
   CLIP(FashionCLIP) vs 시각 유사도 대조군(EfficientNet-B0)을 비교한다.

정량 평가(AUC 등)는 넣지 않는다 — 데이터가 작고(옷장 수백 개, rejected 사례
한 자릿수) 정답 라벨이 없어 통계적 결론을 낼 수 없다. 대신 표·이미지 그리드·
2D 투영으로 정성적으로 관찰한다.

## 실행 전 Kaggle 노트북 설정
- **Accelerator: GPU T4** (Settings → Accelerator)
- **Internet: On** (Settings → Internet — 모델 다운로드에 필요)
- 데이터: `wardrobe_export.zip`을 Private Dataset으로 업로드하고 이 노트북에
  Add Data. 아래 셀은 `/kaggle/input/wardrobe-export/`를 기본 경로로 가정하고,
  없으면 한 겹 아래(`*/items.csv`)를 자동 탐색한다.


## 1. 환경 설정 · 데이터 로드

In [ ]:
!pip -q install open_clip_torch ftfy regex umap-learn --no-warn-script-location


In [ ]:
import itertools
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("경고: GPU가 잡히지 않았습니다 — Settings에서 Accelerator를 GPU T4로 바꿔주세요.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# export.py가 만든 wardrobe_export.zip을 Kaggle Dataset으로 올렸을 때의 기본 경로.
# 데이터셋 이름에 따라 한 겹 더 감싸질 수 있어 items.csv를 기준으로 자동 보정한다.
DATA_DIR = Path("/kaggle/input/wardrobe-export")
if not (DATA_DIR / "items.csv").exists():
    candidates = list(DATA_DIR.glob("*/items.csv"))
    if candidates:
        DATA_DIR = candidates[0].parent
    else:
        raise FileNotFoundError(
            f"items.csv를 찾을 수 없습니다: {DATA_DIR} 아래를 확인하세요. "
            "Add Data에서 wardrobe-export 데이터셋을 붙였는지 확인."
        )
print("DATA_DIR:", DATA_DIR)
IMAGES_DIR = DATA_DIR / "images"

WORK_DIR = Path("/kaggle/working")


In [ ]:
# items.csv 컬럼: itemId, category, subCategory, createdAt, imageSource,
# hasImageFile, color, style, pattern, formality, fit, tags(세미콜론 구분)
items = pd.read_csv(DATA_DIR / "items.csv")

# CSV에는 bool이 "True"/"False" 문자열로 저장되므로 명시적으로 변환한다
# (판다스 버전에 따라 자동 추론에 기대지 않는다).
items["hasImageFile"] = items["hasImageFile"].astype(str).str.strip().str.lower() == "true"
items["tags"] = items["tags"].fillna("").apply(lambda s: [t for t in s.split(";") if t])
for col in ["color", "style", "pattern", "formality", "fit", "subCategory", "category"]:
    items[col] = items[col].fillna("")

with open(DATA_DIR / "history.json", encoding="utf-8") as f:
    history = json.load(f)

print(f"items.csv: {len(items)}행")
print(items["category"].value_counts())

print(f"\nhistory.json: {len(history)}건")
choice_counts = pd.Series([h.get("userChoice") or "pending" for h in history]).value_counts()
print(choice_counts)

items_with_image = items[items["hasImageFile"]].reset_index(drop=True)
print(f"\n이미지 파일 있는 아이템: {len(items_with_image)} / {len(items)}")

item_ids = items_with_image["itemId"].tolist()

attrs_by_id = items_with_image.set_index("itemId")[
    ["category", "color", "style", "pattern", "formality", "fit", "tags"]
].to_dict("index")

def load_rgb(item_id: str) -> Image.Image:
    return Image.open(IMAGES_DIR / f"{item_id}.jpg").convert("RGB")

items_with_image.head()


## 2. 임베딩 추출 (3종, 각각 L2 정규화)

- **OpenCLIP ViT-B/32** (laion2b_s34b_b79k, MIT) — 범용 CLIP
- **FashionCLIP 2.0** (patrickjohncyh/fashion-clip, MIT) — 패션 특화, CLIP과
  동일 아키텍처를 옷 이미지+설명으로 fine-tune
- **EfficientNet-B0 pooled feature** (ImageNet 사전학습, torchvision) — 분류
  head를 떼고 `avgpool` 출력(1280-dim)만 사용하는 시각 유사도 대조군


In [ ]:
def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def load_batch_images(batch_ids):
    imgs, valid_ids = [], []
    for iid in batch_ids:
        try:
            imgs.append(load_rgb(iid))
            valid_ids.append(iid)
        except Exception as e:
            print(f"  ! {iid}: 이미지 로드 실패 — {e}")
    return valid_ids, imgs


In [ ]:
import open_clip

openclip_model, _, openclip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
openclip_model = openclip_model.to(DEVICE).eval()


@torch.no_grad()
def extract_openclip(ids, batch_size=32):
    out = {}
    for batch_ids in batched(ids, batch_size):
        valid_ids, imgs = load_batch_images(batch_ids)
        if not imgs:
            continue
        tensor = torch.stack([openclip_preprocess(im) for im in imgs]).to(DEVICE)
        feats = F.normalize(openclip_model.encode_image(tensor), dim=-1).cpu().numpy()
        out.update(zip(valid_ids, feats))
    return out


print("OpenCLIP 임베딩 추출 중...")
openclip_embeddings = extract_openclip(item_ids)
print(f"  {len(openclip_embeddings)}개 완료, dim={len(next(iter(openclip_embeddings.values())))}")


In [ ]:
from transformers import CLIPModel, CLIPProcessor

fashionclip_model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip").to(DEVICE).eval()
fashionclip_processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip")


@torch.no_grad()
def extract_fashionclip(ids, batch_size=32):
    out = {}
    for batch_ids in batched(ids, batch_size):
        valid_ids, imgs = load_batch_images(batch_ids)
        if not imgs:
            continue
        inputs = fashionclip_processor(images=imgs, return_tensors="pt").to(DEVICE)
        feats = F.normalize(fashionclip_model.get_image_features(**inputs), dim=-1).cpu().numpy()
        out.update(zip(valid_ids, feats))
    return out


print("FashionCLIP 임베딩 추출 중...")
fashionclip_embeddings = extract_fashionclip(item_ids)
print(f"  {len(fashionclip_embeddings)}개 완료, dim={len(next(iter(fashionclip_embeddings.values())))}")


In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

effnet_weights = EfficientNet_B0_Weights.DEFAULT
_effnet_full = efficientnet_b0(weights=effnet_weights).to(DEVICE).eval()
effnet_preprocess = effnet_weights.transforms()

# 분류 head(classifier) 없이 features + avgpool까지만 써서 1280-dim pooled
# feature를 뽑는다 — 이전 온디바이스 스파이크의 EfficientNet-Lite0(TFLite,
# 1280-dim)와 같은 발상의 시각 유사도 대조군을, 여기서는 ImageNet 사전학습
# torchvision B0로 재현한다.
effnet_backbone = torch.nn.Sequential(
    _effnet_full.features, _effnet_full.avgpool, torch.nn.Flatten()
).to(DEVICE).eval()


@torch.no_grad()
def extract_effnet(ids, batch_size=32):
    out = {}
    for batch_ids in batched(ids, batch_size):
        valid_ids, imgs = load_batch_images(batch_ids)
        if not imgs:
            continue
        tensor = torch.stack([effnet_preprocess(im) for im in imgs]).to(DEVICE)
        feats = F.normalize(effnet_backbone(tensor), dim=-1).cpu().numpy()
        out.update(zip(valid_ids, feats))
    return out


print("EfficientNet-B0 pooled feature 추출 중...")
effnet_embeddings = extract_effnet(item_ids)
print(f"  {len(effnet_embeddings)}개 완료, dim={len(next(iter(effnet_embeddings.values())))}")

EMBEDDING_SETS = {
    "effnet": effnet_embeddings,
    "openclip": openclip_embeddings,
    "fashionclip": fashionclip_embeddings,
}
METHOD_LABELS = {
    "tag": "태그",
    "effnet": "EfficientNet",
    "openclip": "OpenCLIP",
    "fashionclip": "FashionCLIP",
}


## 3. 태그 매칭 재구현

`matcher_spec.md`(§2)에 정리된 `OutfitMatcher._compatibilityScore`를 그대로
포팅한다 — 원본은 **color + formality 두 필드만** 사용한다(다른 4개 속성은
매칭 점수에 관여하지 않음).

이 점수 자체는 "카테고리가 다른 두 옷의 코디 궁합"용으로 설계돼 정수형
가중치(-1~4)라 이 노트북의 최근접이웃/프로브 비교(같은 카테고리 아이템끼리도
비교해야 함)에 쓰기엔 해상도가 너무 낮다. 그래서 원본 로직과는 별도로,
6개 속성 전부를 쓰는 **확장 태그 유사도(`tag_similarity`)**를 이 노트북
전용 비교 기준선으로 추가 정의한다 — 앱 로직이 아니라 "태그로만 판단하면
이 정도"를 재는 지표임을 분명히 한다.

In [ ]:
FORMALITY_RANK = {"캐주얼": 0, "세미포멀": 1, "포멀": 2}
NEUTRAL_COLORS = {"화이트", "블랙", "네이비", "그레이", "베이지", "아이보리", "카키", "그레이지"}


def matcher_compatibility_score(a: dict, b: dict) -> float:
    """matcher_spec.md §2 compatibility_score의 리터럴 포팅.
    a, b는 attrs_by_id의 값 형태({'color':..., 'formality':..., ...})."""
    score = 0.0
    rank_a = FORMALITY_RANK.get(a["formality"])
    rank_b = FORMALITY_RANK.get(b["formality"])
    if rank_a is not None and rank_b is not None:
        diff = abs(rank_a - rank_b)
        score += 2 if diff == 0 else (1 if diff == 1 else -1)
    if a["color"] in NEUTRAL_COLORS or b["color"] in NEUTRAL_COLORS:
        score += 2
    elif a["color"] == b["color"] and a["color"] != "":
        score += 1
    return score


def tag_similarity(a: dict, b: dict) -> float:
    """이 노트북 전용 확장 태그 유사도(0~6). color/style/pattern/formality/fit
    5개 필드의 완전 일치 개수 + tags 리스트의 Jaccard 유사도. matcher의
    실제 매칭 로직이 아니라, '태그만으로 얼마나 비슷한가'를 재는 비교 기준선."""
    fields = ["color", "style", "pattern", "formality", "fit"]
    fields_equal = sum(1.0 for f in fields if a[f] == b[f] and a[f] != "")
    tags_a, tags_b = set(a["tags"]), set(b["tags"])
    union = tags_a | tags_b
    jaccard = len(tags_a & tags_b) / len(union) if union else 0.0
    return fields_equal + jaccard


def cosine(u, v) -> float:
    return float(np.dot(u, v))  # 두 벡터 모두 이미 L2 정규화됨


def similarity(method: str, id_a: str, id_b: str) -> float:
    if method == "tag":
        return tag_similarity(attrs_by_id[id_a], attrs_by_id[id_b])
    return cosine(EMBEDDING_SETS[method][id_a], EMBEDDING_SETS[method][id_b])


# 빠른 확인: 원본 matcher 점수와 확장 태그 유사도가 같은 방향으로 움직이는지
_sample_a, _sample_b = item_ids[0], item_ids[min(1, len(item_ids) - 1)]
print("matcher_compatibility_score 예시:", matcher_compatibility_score(attrs_by_id[_sample_a], attrs_by_id[_sample_b]))
print("tag_similarity 예시:", tag_similarity(attrs_by_id[_sample_a], attrs_by_id[_sample_b]))


## 4. 프로브 테스트 (가장 중요)

핵심 질문: **태그 매칭이 구분 못 하는 걸 임베딩이 갈라내는가?**

- **(a) 같은 태그 · 다른 재질**: category+color+style+pattern+formality+fit이
  전부 동일한(=태그로는 완전히 같은 아이템으로 보이는) 아이템 쌍을 자동으로
  찾는다. `tag_sim`은 항상 최고치로 동일할 텐데, 임베딩 유사도가 그만큼
  높지 않다면(변별력이 있다면) 임베딩이 태그가 못 보는 재질/디테일 차이를
  잡아낸다는 뜻이다.
- **(b) 다른 태그 · 비슷한 무드**: color 또는 formality가 달라(`tag_sim`이
  낮아) 태그상으로는 안 어울려 보이지만, FashionCLIP 유사도 기준 상위인
  같은 카테고리 쌍을 찾는다. `tag_sim`은 낮은데 임베딩 유사도가 높다면,
  임베딩이 태그가 놓치는 시각적 유사성을 찾아낸다는 뜻이다.

In [ ]:
def probe_table(pairs, title):
    rows = []
    for a, b in pairs:
        rows.append({
            "item_a": a,
            "item_b": b,
            "tag_sim": round(similarity("tag", a, b), 3),
            "effnet_sim": round(similarity("effnet", a, b), 3),
            "openclip_sim": round(similarity("openclip", a, b), 3),
            "fashionclip_sim": round(similarity("fashionclip", a, b), 3),
        })
    df = pd.DataFrame(rows)
    print(title)
    display(df)
    return df


def show_pairs(pairs, max_pairs=10):
    pairs = pairs[:max_pairs]
    if not pairs:
        print("(표시할 쌍이 없음)")
        return
    fig, axes = plt.subplots(len(pairs), 2, figsize=(4, 2.2 * len(pairs)))
    if len(pairs) == 1:
        axes = axes.reshape(1, 2)
    for row, (a, b) in enumerate(pairs):
        for col, iid in enumerate((a, b)):
            ax = axes[row, col]
            try:
                ax.imshow(load_rgb(iid))
            except Exception:
                pass
            ax.set_title(iid, fontsize=8)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# (a) 같은 태그 · 다른 재질 — 자동 탐색
group_cols = ["category", "color", "style", "pattern", "formality", "fit"]
groups = items_with_image.groupby(group_cols)["itemId"].apply(list)
same_tag_groups = [ids for ids in groups if len(ids) >= 2]
print(f"완전히 같은 태그 조합을 가진 그룹: {len(same_tag_groups)}개")

random.shuffle(same_tag_groups)
same_tag_pairs = []
for ids in same_tag_groups:
    same_tag_pairs.extend(itertools.combinations(ids, 2))
    if len(same_tag_pairs) >= 10:
        break
same_tag_pairs = same_tag_pairs[:10]
print(f"프로브 (a) 후보 쌍: {len(same_tag_pairs)}개 (데이터가 적으면 5개 미만일 수 있음)")

same_tag_df = probe_table(same_tag_pairs, "프로브 (a) 같은 태그 · 다른 재질")
show_pairs(same_tag_pairs)


In [ ]:
# (b) 다른 태그 · 비슷한 무드 — FashionCLIP 유사도 상위로 자동 탐색
diff_tag_scored = []
for a, b in itertools.combinations(item_ids, 2):
    aa, bb = attrs_by_id[a], attrs_by_id[b]
    if aa["category"] != bb["category"]:
        continue  # 카테고리가 다르면 '무드가 비슷하다'는 비교 자체가 어색함
    if aa["color"] == bb["color"] and aa["formality"] == bb["formality"]:
        continue  # 태그가 이미 같으면 (a)번과 겹치므로 제외
    diff_tag_scored.append((a, b, similarity("fashionclip", a, b)))

diff_tag_scored.sort(key=lambda t: t[2], reverse=True)
diff_tag_pairs = [(a, b) for a, b, _ in diff_tag_scored[:10]]
print(f"프로브 (b) 후보 쌍: {len(diff_tag_pairs)}개")

diff_tag_df = probe_table(diff_tag_pairs, "프로브 (b) 다른 태그 · 비슷한 무드 (FashionCLIP 유사도 상위)")
show_pairs(diff_tag_pairs)


## 5. 최근접 이웃 정성 평가

쿼리 10개(랜덤, seed 고정) × 방법 4가지(태그/EfficientNet/OpenCLIP/
FashionCLIP)의 top-5 최근접 이웃을 이미지 그리드로 비교한다. 맨 위 행은
쿼리 이미지, 그 아래 4행이 각 방법의 top-5.

In [ ]:
def top_k_neighbors(method, query_id, k=5):
    scores = [(cand, similarity(method, query_id, cand)) for cand in item_ids if cand != query_id]
    scores.sort(key=lambda t: t[1], reverse=True)
    return [c for c, _ in scores[:k]]


METHOD_ORDER = ["tag", "effnet", "openclip", "fashionclip"]
query_ids = random.sample(item_ids, min(10, len(item_ids)))

for qid in query_ids:
    fig, axes = plt.subplots(len(METHOD_ORDER) + 1, 5, figsize=(10, 2.1 * (len(METHOD_ORDER) + 1)))
    for ax in axes[0]:
        ax.axis("off")
    mid_col = 5 // 2
    try:
        axes[0, mid_col].imshow(load_rgb(qid))
    except Exception:
        pass
    axes[0, mid_col].set_title(f"쿼리: {qid}", fontsize=9)

    for r, method in enumerate(METHOD_ORDER, start=1):
        neighbors = top_k_neighbors(method, qid)
        for c in range(5):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(neighbors):
                nid = neighbors[c]
                try:
                    ax.imshow(load_rgb(nid))
                except Exception:
                    pass
                ax.set_title(f"{METHOD_LABELS[method]}\n{nid}", fontsize=7)
    plt.tight_layout()
    plt.show()


## 6. 임베딩 공간 구조

UMAP(설치 실패 시 t-SNE로 폴백)으로 2D 투영, 3개 임베딩 모델 각각을
카테고리별/색상별로 색칠해 비교한다.

In [ ]:
try:
    import umap

    def project_2d(X):
        return umap.UMAP(random_state=SEED).fit_transform(X)

    projection_method = "UMAP"
except ImportError:
    from sklearn.manifold import TSNE

    def project_2d(X):
        return TSNE(random_state=SEED, init="pca").fit_transform(X)

    projection_method = "t-SNE (umap-learn 설치 실패로 폴백)"

print(f"2D 투영 방법: {projection_method}")

categories_list = [attrs_by_id[i]["category"] for i in item_ids]
colors_list = [attrs_by_id[i]["color"] for i in item_ids]

fig, axes = plt.subplots(3, 2, figsize=(12, 15))
for row, method in enumerate(["effnet", "openclip", "fashionclip"]):
    X = np.stack([EMBEDDING_SETS[method][i] for i in item_ids])
    coords = project_2d(X)
    for col, (label_list, title) in enumerate([(categories_list, "카테고리"), (colors_list, "색상")]):
        ax = axes[row, col]
        uniq = sorted(set(label_list))
        cmap = plt.get_cmap("tab20", max(len(uniq), 1))
        for i, lab in enumerate(uniq):
            idxs = [j for j, l in enumerate(label_list) if l == lab]
            ax.scatter(coords[idxs, 0], coords[idxs, 1], s=14, color=cmap(i), label=lab or "(미상)")
        ax.set_title(f"{METHOD_LABELS[method]} · {title}별")
        if row == 0:
            ax.legend(fontsize=6, markerscale=0.8, loc="best")
plt.tight_layout()
plt.show()


## 7. rejected 사례 정성 분석

`history.json`에서 `userChoice == "rejected_with_alternative"`인 사례 —
추천한 조합 대신 사용자가 실제로 다른 조합을 고른 경우. **건수가 적어
(보통 한 자릿수) 통계가 아니라 사례 관찰이다.** 조합 임베딩(아이템 임베딩의
평균, 재정규화)끼리의 코사인 유사도로 "방향은 맞고 세부만 틀렸는지"
(유사도 높음) "방향 자체가 틀렸는지"(유사도 낮음)를 가늠한다 — 0.7이라는
경계값은 엄밀한 기준이 아니라 대략적인 참고선이다.

In [ ]:
def combo_embedding(method, ids):
    vecs = [EMBEDDING_SETS[method][i] for i in ids if i in EMBEDDING_SETS[method]]
    if not vecs:
        return None
    v = np.mean(vecs, axis=0)
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v


rejected = [h for h in history if h.get("userChoice") == "rejected_with_alternative"]
print(f"rejected_with_alternative 사례: {len(rejected)}건 — 통계가 아니라 사례 관찰로 읽을 것")

for idx, h in enumerate(rejected, start=1):
    rec_ids = h.get("itemIds", []) or []
    chosen_ids = h.get("userChosenItemIds", []) or []
    print(f"\n=== rejected 사례 {idx}/{len(rejected)} — [{h.get('targetTpoTag') or '일상'}] {h.get('createdAt')} ===")
    print(f"추천 조합: {rec_ids}")
    print(f"실제 선택: {chosen_ids}")

    max_len = max(len(rec_ids), len(chosen_ids), 1)
    fig, axes = plt.subplots(2, max_len, figsize=(2.2 * max_len, 4.6))
    if max_len == 1:
        axes = axes.reshape(2, 1)
    for col in range(max_len):
        ax_rec, ax_chosen = axes[0, col], axes[1, col]
        ax_rec.axis("off")
        ax_chosen.axis("off")
        if col < len(rec_ids):
            try:
                ax_rec.imshow(load_rgb(rec_ids[col]))
            except Exception:
                pass
        if col == 0:
            ax_rec.set_title("추천", fontsize=9)
        if col < len(chosen_ids):
            try:
                ax_chosen.imshow(load_rgb(chosen_ids[col]))
            except Exception:
                pass
        if col == 0:
            ax_chosen.set_title("실제 선택", fontsize=9)
    plt.tight_layout()
    plt.show()

    for method in ["effnet", "openclip", "fashionclip"]:
        rec_vec = combo_embedding(method, rec_ids)
        chosen_vec = combo_embedding(method, chosen_ids)
        if rec_vec is None or chosen_vec is None:
            print(f"  {METHOD_LABELS[method]}: 이미지 누락으로 조합 임베딩 계산 불가 — 스킵")
            continue
        sim = float(np.dot(rec_vec, chosen_vec))
        verdict = "가까움 → 방향은 맞고 세부가 틀림" if sim > 0.7 else "멀음 → 방향 자체가 틀림"
        print(f"  {METHOD_LABELS[method]}: 조합 유사도 {sim:.3f} ({verdict})")


## 8. 결과 내보내기

위 관찰을 바탕으로 채택 후보 모델 하나를 골라 `embeddings.json`으로 저장한다
(나중에 Firestore 백필 스크립트에서 `itemId -> 벡터`로 그대로 사용). 기본값은
`fashionclip`이지만, 6~7번 결과를 보고 `CHOSEN_MODEL`을 바꿔 이 셀만 다시
실행해도 된다.

In [ ]:
CHOSEN_MODEL = "fashionclip"  # 6~7번 결과를 본 뒤 필요하면 "openclip"/"effnet"으로 변경

export_embeddings = {iid: vec.tolist() for iid, vec in EMBEDDING_SETS[CHOSEN_MODEL].items()}
embeddings_path = WORK_DIR / "embeddings.json"
with open(embeddings_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": CHOSEN_MODEL,
            "dim": len(next(iter(export_embeddings.values()))),
            "embeddings": export_embeddings,
        },
        f,
    )
print(f"{embeddings_path} 저장 완료 — {len(export_embeddings)}개 아이템, model={CHOSEN_MODEL}")

summary_rows = [
    {
        "model": METHOD_LABELS[m],
        "dim": len(next(iter(EMBEDDING_SETS[m].values()))),
        "n_items": len(EMBEDDING_SETS[m]),
    }
    for m in ["effnet", "openclip", "fashionclip"]
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_path = WORK_DIR / "comparison_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"{summary_path} 저장 완료")
